In [0]:
# ============================================================
# Bronze — Source 08: Shopify GraphQL
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=08_shopify/
# Target: bronze.shopify catalog in Unity Catalog
#
# Tables:
#   - bronze.shopify.products
#
# Raw format: Parquet (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "08_shopify"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/")

In [0]:
# Cell 2 — Read and inspect both file types
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path_products = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/shopify_products_*.json"
path_discounts = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/shopify_discounts_*.json"

df_products = spark.read.format("json").load(path_products) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

df_discounts = spark.read.format("json").load(path_discounts) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

print(f"Products files count: {df_products.count()}")
print(f"Discounts files count: {df_discounts.count()}")
df_products.printSchema()

In [0]:
# Cell 3 — Flatten products correctly
from pyspark.sql.functions import explode

products_flat = df_products \
    .select(explode(col("edges")).alias("edge")) \
    .select(
        col("edge.node.id").alias("shopify_id"),
        col("edge.node.handle").alias("handle"),
        col("edge.node.productType").alias("product_type"),
        col("edge.node.status"),
        col("edge.node._source").alias("source"),
    )

print(f"Flattened products: {products_flat.count()} rows")
products_flat.show(3)

In [0]:
# Cell 4 — Flatten discounts with correct fields
discounts_flat = df_discounts \
    .select(explode(col("edges")).alias("edge")) \
    .select(
        col("edge.node.id").alias("shopify_id"),
        col("edge.node.title"),
        col("edge.node.code"),
        col("edge.node.status"),
        col("edge.node.percentage_off"),
        col("edge.node.usage_count"),
        col("edge.node.applies_once_per_customer"),
        col("edge.node.created_at"),
        col("edge.node._source").alias("source"),
    )

print(f"Flattened discounts: {discounts_flat.count()} rows")
discounts_flat.show(3)

In [0]:
# Cell 5 — Write both tables to Bronze and update watermark
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.shopify")

products_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.shopify.products")

discounts_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.shopify.discounts")

latest_ts = df_products.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, products_flat.count() + discounts_flat.count())

print(f"✅ bronze.shopify.products: {products_flat.count()} rows")
print(f"✅ bronze.shopify.discounts: {discounts_flat.count()} rows")

In [0]:
print(f"bronze.shopify.products: {spark.sql('SELECT COUNT(*) as cnt FROM bronze.shopify.products').collect()[0][0]} rows")
print(f"bronze.shopify.discounts: {spark.sql('SELECT COUNT(*) as cnt FROM bronze.shopify.discounts').collect()[0][0]} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '08_shopify'").show()